# Aula 07 — Laboratório
## Quem se parece com quem

**Análise Econômica e Machine Learning** · Prof. Dr. Flavius Sodré · Economia, 8º período

Até aqui, todo modelo tinha uma resposta para prever e conferir. Agora não há: só as
colunas. Neste laboratório:

1. aquecimento com clientes simulados, onde os segmentos verdadeiros são conhecidos;
2. os 185 municípios de Pernambuco, da planilha à matriz padronizada;
3. componentes principais e um índice de condições de vida;
4. cinco grupos de municípios, com nome, perfil e mapa;
5. a robustez dos grupos: sementes, outro algoritmo, uma variável a menos.

**Como completar:** onde houver `# COMPLETE`, escreva o código. Onde houver `___`,
substitua. Rode as células na ordem.

## Bloco 0 — ambiente

In [ ]:
# Bloco 0 — rode esta celula e confira a saida.
import os, sys, json, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 140)
pd.set_option("display.precision", 2)
plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# As mesmas cores dos slides. Clientes: azul = esporadicos, laranja =
# frequentes, verde = alta renda. Municipios: azul = polos urbanos, laranja =
# enclaves industriais, verde = base agropecuaria, cinzas = os dois interiores.
AZUL, LARANJA, VERDE, BORDO = "#2a78d6", "#eb6834", "#1baf7a", "#6F1D2B"
CINZA, CINZA_CLARO = "#8A8D96", "#c9cbd1"

try:
    import sklearn
    from sklearn.cluster import KMeans
    from sklearn.decomposition import PCA
    from sklearn.metrics import adjusted_rand_score, silhouette_score
    from sklearn.preprocessing import StandardScaler
except ImportError:
    raise ImportError("Este laboratorio precisa do scikit-learn. "
                      "Instale com:  pip install scikit-learn")
import scipy
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.optimize import linear_sum_assignment

print("Python      :", sys.version.split()[0], "|", platform.system())
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("scipy       :", scipy.__version__)
print("\nAmbiente pronto.")

## Bloco 1 — aquecimento com a verdade na mão

Uma rede de varejo tem 600 clientes. De cada um, sabe a renda mensal (R$) e quantas
compras faz por mês. Os clientes pertencem a três segmentos — esporádicos, frequentes e
alta renda —, mas a empresa não sabe disso: a coluna `segmento` existe só para conferir.

In [ ]:
# Bloco 1 — as ferramentas. Leia, nao precisa alterar.

SEMENTE = 2026
N_CLIENTES = 600
SEGMENTOS = ['Esporádicos', 'Frequentes', 'Alta renda']
SEG_MEDIA = np.array([[2600.0, 3.0], [2800.0, 10.0], [7200.0, 6.5]])   # renda (R$), compras por mes
SEG_DP = np.array([[750.0, 1.0], [750.0, 1.2], [1300.0, 1.1]])
CENTROS_INICIAIS = [[-1.1, -1.45], [-0.75, -1.05], [-1.35, -0.85]]

PERFIL = ['urbanizacao', 'alfabetizacao', 'renda_pc', 'pobreza', 'esgoto', 'agua', 'pib_pc',
 'agropecuaria', 'industria', 'adm_publica', 'emprego_formal', 'envelhecimento']
EM_LOG = ['renda_pc', 'pib_pc', 'emprego_formal']
INDICE = ['renda_pc', 'pobreza', 'alfabetizacao', 'esgoto', 'agua']
GRUPOS = ['Polos urbanos', 'Enclaves industriais', 'Base agropecuária', 'Interior urbanizado', 'Interior rural']
NOMES = {'urbanizacao': 'Urbanização',
 'alfabetizacao': 'Alfabetização',
 'renda_pc': 'Renda per capita',
 'pobreza': 'Pobreza',
 'esgoto': 'Esgoto',
 'agua': 'Água',
 'pib_pc': 'PIB per capita',
 'agropecuaria': 'Agropecuária',
 'industria': 'Indústria',
 'adm_publica': 'Administração pública',
 'emprego_formal': 'Emprego formal',
 'envelhecimento': 'Envelhecimento',
 'populacao': 'População',
 'densidade': 'Densidade',
 'servicos': 'Serviços',
 'salario_medio': 'Salário médio',
 'idade_mediana': 'Idade mediana'}


def clientes(n=N_CLIENTES, semente=SEMENTE):
    """Clientes de uma rede de varejo em três segmentos: renda mensal (R$) e
    compras por mês (média do último ano). O segmento é a resposta certa — o
    algoritmo não o vê."""
    rng = np.random.default_rng(semente)
    seg = np.repeat(np.arange(3), n // 3)
    X = SEG_MEDIA[seg] + SEG_DP[seg] * rng.normal(size=(len(seg), 2))
    return pd.DataFrame({"renda": np.round(np.clip(X[:, 0], 800, None), -1),
                         "compras": np.round(np.clip(X[:, 1], 0.2, None), 1),
                         "segmento": np.array(SEGMENTOS)[seg]})


def nuvem(n=N_CLIENTES, semente=SEMENTE):
    """Uma clientela sem segmentos: uma só nuvem, com a média e a dispersão
    totais dos clientes simulados."""
    ref = clientes()
    rng = np.random.default_rng(semente + 1)
    renda = ref["renda"].mean() + ref["renda"].std() * rng.normal(size=n)
    compras = ref["compras"].mean() + ref["compras"].std() * rng.normal(size=n)
    return pd.DataFrame({"renda": np.round(np.clip(renda, 800, None), -1),
                         "compras": np.round(np.clip(compras, 0.2, None), 1)})


def padronizar(X):
    """Média zero e desvio-padrão um em cada coluna."""
    return pd.DataFrame(StandardScaler().fit_transform(X), index=X.index, columns=X.columns)


def kmeans(Z, k, semente=0, n_init=50):
    return KMeans(n_clusters=k, n_init=n_init, random_state=semente).fit(np.asarray(Z))


def curva_k(Z, ks=range(2, 9)):
    """Inércia (soma das distâncias ao quadrado até o centro) e silhueta para cada k."""
    Z = np.asarray(Z)
    saida = {}
    for k in ks:
        km = kmeans(Z, k)
        saida[k] = {"inercia": float(km.inertia_),
                    "silhueta": float(silhouette_score(Z, km.labels_))}
    return saida


def acerto(verdade, grupos):
    """Fração classificada no segmento certo, depois de casar cada grupo com o
    segmento em que ele mais acerta (os números dos grupos são arbitrários)."""
    t = pd.crosstab(np.asarray(grupos), np.asarray(verdade)).to_numpy()
    lin, col = linear_sum_assignment(-t)
    return float(t[lin, col].sum() / t.sum())


def kmeans_passo_a_passo(Z, centros, max_iter=50):
    """K-means escrito à mão: atribui cada ponto ao centro mais próximo e leva
    cada centro para a média do seu grupo, até nada mudar."""
    Z = np.asarray(Z)
    c = np.asarray(centros, dtype=float).copy()
    passos = []
    rot = None
    for _ in range(max_iter):
        d = ((Z[:, None, :] - c[None, :, :]) ** 2).sum(axis=2)
        novo = d.argmin(axis=1)
        passos.append({"centros": c.copy(), "rotulos": novo.copy()})
        if rot is not None and np.array_equal(novo, rot):
            break
        rot = novo
        c = np.array([Z[rot == j].mean(axis=0) for j in range(len(c))])
    return passos


print("Ferramentas prontas.")

In [ ]:
# Bloco 1 — o K-means escrito a mao. Rode e olhe as quatro figuras.
cl = clientes()
X = cl[["renda", "compras"]]
media, dp = X.mean().to_numpy(), X.std(ddof=0).to_numpy()
passos = kmeans_passo_a_passo(padronizar(X), CENTROS_INICIAIS)
print(f"{len(passos)} atribuicoes ate nada mudar")

fig, axs = plt.subplots(1, 4, figsize=(15, 3.8), sharey=True)
for ax, i in zip(axs, [0, 1, 2, len(passos) - 1]):
    c = passos[i]["centros"] * dp + media
    ax.scatter(X["renda"] / 1000, X["compras"], s=6, alpha=0.5,
               c=[[AZUL, LARANJA, VERDE][r] for r in passos[i]["rotulos"]])
    ax.scatter(c[:, 0] / 1000, c[:, 1], s=200, marker="X", c="k")
    ax.set_title(f"rodada {i + 1}"); ax.set_xlabel("renda (R$ mil)")
axs[0].set_ylabel("compras por mes")
plt.show()

In [ ]:
# Bloco 1 — padronizar ou nao. COMPLETE.
#
# Rode o K-means com k = 3 nas duas colunas como estao (X) e padronizadas, e
# compare cada resultado com os segmentos verdadeiros usando acerto().

# COMPLETE
km_bruto = ___
km_padr = ___

print(f"sem padronizar: acerta {100 * acerto(cl['segmento'], km_bruto.labels_):.1f}%")
print(f"padronizado   : acerta {100 * acerto(cl['segmento'], km_padr.labels_):.1f}%")

fig, axs = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, km, tit in ((axs[0], km_bruto, "sem padronizar"), (axs[1], km_padr, "padronizado")):
    ax.scatter(X["renda"] / 1000, X["compras"], s=8,
               c=[[AZUL, LARANJA, VERDE][r] for r in km.labels_])
    ax.set_title(tit); ax.set_xlabel("renda (R$ mil)")
axs[0].set_ylabel("compras por mes")
plt.show()

In [ ]:
# Bloco 1 — quantos grupos? COMPLETE.
#
# Calcule a curva de k (inercia e silhueta de k = 2 a 8) para os clientes
# padronizados e para uma clientela sem segmentos, gerada por nuvem().

# COMPLETE
curva_cl = ___
curva_nv = ___

ks = list(curva_cl)
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].plot(ks, [curva_cl[k]["inercia"] for k in ks], "o-", color=AZUL)
axs[0].set_title("inercia (clientes)"); axs[0].set_xlabel("k")
axs[1].plot(ks, [curva_cl[k]["silhueta"] for k in ks], "o-", color=AZUL, label="tres segmentos")
axs[1].plot(ks, [curva_nv[k]["silhueta"] for k in ks], "o--", color=CINZA, label="uma nuvem so")
axs[1].set_title("silhueta"); axs[1].set_xlabel("k"); axs[1].legend()
plt.show()
print(pd.DataFrame({"silhueta_clientes": {k: v["silhueta"] for k, v in curva_cl.items()},
                    "silhueta_nuvem": {k: v["silhueta"] for k, v in curva_nv.items()}}).round(3))

### Responda

> Na clientela sem segmentos, o K-means com k = 3 também devolve três grupos. O que a
> silhueta mostra que a lista de grupos não mostra?

## Bloco 2 — da planilha à matriz

Os 185 municípios de Pernambuco (184 municípios e o distrito estadual de Fernando de
Noronha), com dados do IBGE. Para o perfil socioeconômico, doze indicadores:
urbanização, alfabetização, renda per capita, pobreza, esgoto, água, PIB per capita,
peso da agropecuária, da indústria e da administração pública no valor adicionado,
emprego formal e envelhecimento.

In [ ]:
# Bloco 2 — Plano B. Rode e siga em frente; nao precisa ler.
#
# Resumo da base real (quantos municipios ha em cada tipo de economia e em
# cada mesorregiao, e a media e o desvio de cada indicador dentro do tipo).
# Serve so para gerar municipios inventados com a mesma estrutura.

MESORREGIOES = {1: 'Metropolitana do Recife',
 2: 'Mata Pernambucana',
 3: 'Agreste Pernambucano',
 4: 'Sertão Pernambucano',
 5: 'São Francisco Pernambucano'}
COLUNAS = ['cod_ibge', 'municipio', 'mesorregiao', 'populacao', 'area_km2', 'densidade',
 'urbanizacao', 'alfabetizacao', 'envelhecimento', 'idade_mediana', 'renda_pc', 'pobreza',
 'esgoto', 'agua', 'pib_pc', 'agropecuaria', 'industria', 'servicos', 'adm_publica',
 'emprego_formal', 'salario_medio']
TRANSF = {'populacao': 'log',
 'area_km2': 'log',
 'urbanizacao': 'pct',
 'alfabetizacao': 'pct',
 'envelhecimento': 'log',
 'idade_mediana': 'id',
 'renda_pc': 'log',
 'pobreza': 'pct',
 'esgoto': 'pct',
 'agua': 'pct',
 'pib_pc': 'log',
 'agropecuaria': 'pct',
 'industria': 'pct',
 'adm_publica': 'pct',
 'emprego_formal': 'log',
 'salario_medio': 'id'}

ESP_PLANO_B = {'Polos urbanos': {'n': 31,
                   'meso': {'Metropolitana do Recife': 9,
                            'Mata Pernambucana': 8,
                            'Agreste Pernambucano': 8,
                            'Sertão Pernambucano': 4,
                            'São Francisco Pernambucano': 2},
                   'v': {'populacao': [11.349, 1.149],
                         'area_km2': [5.606, 1.274],
                         'urbanizacao': [89.508, 8.081],
                         'alfabetizacao': [87.476, 4.178],
                         'envelhecimento': [4.194, 0.265],
                         'idade_mediana': [33.452, 2.434],
                         'renda_pc': [6.929, 0.265],
                         'pobreza': [19.854, 4.504],
                         'esgoto': [57.318, 20.78],
                         'agua': [75.903, 11.543],
                         'pib_pc': [9.972, 0.369],
                         'agropecuaria': [3.24, 2.811],
                         'industria': [14.104, 7.114],
                         'adm_publica': [30.344, 7.926],
                         'emprego_formal': [2.751, 0.414],
                         'salario_medio': [1.761, 0.328]}},
 'Enclaves industriais': {'n': 9,
                          'meso': {'Metropolitana do Recife': 4,
                                   'Mata Pernambucana': 3,
                                   'Agreste Pernambucano': 1,
                                   'Sertão Pernambucano': 0,
                                   'São Francisco Pernambucano': 1},
                          'v': {'populacao': [10.827, 0.971],
                                'area_km2': [5.513, 1.125],
                                'urbanizacao': [88.261, 7.629],
                                'alfabetizacao': [85.379, 4.023],
                                'envelhecimento': [4.072, 0.179],
                                'idade_mediana': [32.222, 1.685],
                                'renda_pc': [6.691, 0.103],
                                'pobreza': [23.532, 2.496],
                                'esgoto': [35.562, 21.948],
                                'agua': [67.934, 17.887],
                                'pib_pc': [11.032, 0.666],
                                'agropecuaria': [3.763, 2.459],
                                'industria': [51.04, 11.762],
                                'adm_publica': [14.109, 6.424],
                                'emprego_formal': [3.181, 0.506],
                                'salario_medio': [2.211, 0.534]}},
 'Base agropecuária': {'n': 19,
                       'meso': {'Metropolitana do Recife': 0,
                                'Mata Pernambucana': 8,
                                'Agreste Pernambucano': 7,
                                'Sertão Pernambucano': 0,
                                'São Francisco Pernambucano': 4},
                       'v': {'populacao': [9.95, 0.528],
                             'area_km2': [5.693, 1.051],
                             'urbanizacao': [64.165, 15.509],
                             'alfabetizacao': [79.746, 2.728],
                             'envelhecimento': [4.075, 0.237],
                             'idade_mediana': [31.684, 2.104],
                             'renda_pc': [6.59, 0.13],
                             'pobreza': [26.939, 4.304],
                             'esgoto': [35.455, 15.108],
                             'agua': [64.116, 15.696],
                             'pib_pc': [9.842, 0.259],
                             'agropecuaria': [35.521, 12.101],
                             'industria': [7.787, 7.056],
                             'adm_publica': [33.745, 5.897],
                             'emprego_formal': [2.457, 0.457],
                             'salario_medio': [1.589, 0.18]}},
 'Interior urbanizado': {'n': 71,
                         'meso': {'Metropolitana do Recife': 1,
                                  'Mata Pernambucana': 20,
                                  'Agreste Pernambucano': 29,
                                  'Sertão Pernambucano': 17,
                                  'São Francisco Pernambucano': 4},
                         'v': {'populacao': [9.891, 0.563],
                               'area_km2': [5.599, 0.913],
                               'urbanizacao': [71.002, 11.088],
                               'alfabetizacao': [78.88, 3.723],
                               'envelhecimento': [4.214, 0.163],
                               'idade_mediana': [32.634, 1.629],
                               'renda_pc': [6.59, 0.102],
                               'pobreza': [28.296, 4.34],
                               'esgoto': [54.447, 15.29],
                               'agua': [66.148, 14.348],
                               'pib_pc': [9.484, 0.194],
                               'agropecuaria': [12.499, 7.153],
                               'industria': [7.97, 6.141],
                               'adm_publica': [48.695, 8.005],
                               'emprego_formal': [2.086, 0.293],
                               'salario_medio': [1.632, 0.163]}},
 'Interior rural': {'n': 55,
                    'meso': {'Metropolitana do Recife': 1,
                             'Mata Pernambucana': 4,
                             'Agreste Pernambucano': 26,
                             'Sertão Pernambucano': 20,
                             'São Francisco Pernambucano': 4},
                    'v': {'populacao': [9.638, 0.56],
                          'area_km2': [5.932, 0.988],
                          'urbanizacao': [46.939, 15.969],
                          'alfabetizacao': [76.193, 3.535],
                          'envelhecimento': [4.203, 0.223],
                          'idade_mediana': [32.109, 2.417],
                          'renda_pc': [6.47, 0.128],
                          'pobreza': [34.67, 6.619],
                          'esgoto': [30.843, 14.698],
                          'agua': [38.933, 15.794],
                          'pib_pc': [9.315, 0.181],
                          'agropecuaria': [12.495, 7.151],
                          'industria': [6.615, 9.648],
                          'adm_publica': [55.896, 9.709],
                          'emprego_formal': [1.875, 0.282],
                          'salario_medio': [1.64, 0.211]}}}


def dados_sinteticos(ESP, semente=SEMENTE):
    """Plano B: 185 municípios INVENTADOS, com as mesmas colunas e os mesmos
    cinco tipos de economia da base real. Cada indicador é sorteado em torno da
    média do seu tipo; serviços fecham os 100% do valor adicionado."""
    rng = np.random.default_rng(semente)
    linhas = []
    for g, e in ESP.items():
        mesos = [m for m, n in e["meso"].items() for _ in range(n)]
        for i in range(e["n"]):
            r = {}
            for c, (m, s) in e["v"].items():
                x = m + s * rng.normal()
                t = TRANSF[c]
                r[c] = float(np.exp(x)) if t == "log" else float(x)
            for c in ("urbanizacao", "alfabetizacao", "pobreza", "esgoto", "agua"):
                r[c] = float(np.clip(r[c], 1.0, 100.0))
            for c in ("agropecuaria", "industria", "adm_publica"):
                r[c] = float(np.clip(r[c], 0.1, 90.0))
            tot = r["agropecuaria"] + r["industria"] + r["adm_publica"]
            if tot > 95:
                for c in ("agropecuaria", "industria", "adm_publica"):
                    r[c] *= 95 / tot
            r["servicos"] = 100 - r["agropecuaria"] - r["industria"] - r["adm_publica"]
            r["mesorregiao"] = mesos[i]
            linhas.append(r)
    d = pd.DataFrame(linhas).sample(frac=1, random_state=semente).reset_index(drop=True)
    d["cod_ibge"] = 9900001 + d.index
    d["municipio"] = [f"Município sintético {i + 1:03d}" for i in d.index]
    d["populacao"] = d["populacao"].round().astype(int)
    d["idade_mediana"] = d["idade_mediana"].round().astype(int)
    d["densidade"] = d["populacao"] / d["area_km2"]
    for c in COLUNAS:
        if d[c].dtype == float:
            d[c] = d[c].round(1 if c == "salario_medio" else 2)
    d["area_km2"] = d["area_km2"].round(3)
    return d[COLUNAS]


print("Plano B pronto (so e usado se os dados reais nao carregarem).")

In [ ]:
# Bloco 2 — os dados. Tres caminhos, nesta ordem:
#   1. o arquivo municipios_pe.csv, na pasta Dados do repositorio;
#   2. a API de agregados do IBGE, refazendo a conta a partir das mesmas
#      tabelas (Censo 2022, PIB dos Municipios, Cadastro Central de Empresas)
#      — lenta: ate 2 minutos por tabela;
#   3. o PLANO B (celula acima), com dados inventados.
# Cada chamada tenta de novo desde o comeco: se a rede voltar, basta rodar
# a celula outra vez.
import requests

ARQ_CSV = "../Dados/municipios_pe.csv"
MESO_API = {'Agreste Pernambucano': 3,
 'Mata Pernambucana': 2,
 'Metropolitana de Recife': 1,
 'Sertão Pernambucano': 4,
 'São Francisco Pernambucano': 5}
COLUNAS_BRUTAS = ['cod', 'nome', 'meso', 'pop', 'area', 'urb', 'alfab', 'env', 'idmed', 'rmedia',
 'rmediana', 'mtot', 'm14', 'm12', 'msem', 'dtot', 'desg', 'datot', 'dagua', 'pib22',
 'pib21', 'vab21', 'agro21', 'ind21', 'serv21', 'adm21', 'assal22', 'salsm22', 'salrs22']
API = 'https://servicodados.ibge.gov.br/api/v3/agregados'
API_LOCALIDADES = 'https://servicodados.ibge.gov.br/api/v1/localidades/estados/26/municipios'
CHAMADAS = [(4714, 2022, '93|6318', '', {'area': ('6318', ''), 'pop': ('93', '')}),
 (10211, 2022, '1000093', '2661[32776]|1[1]', {'urb': ('1000093', '32776|1')}),
 (10091, 2022, '2513', '2[6794]|58[95253]|2661[32776]|1[6795]',
  {'alfab': ('2513', '6794|95253|32776|6795')}),
 (10097, 2022, '9175|10613', '2661[32776]|1[6795]',
  {'env': ('9175', '32776|6795'), 'idmed': ('10613', '32776|6795')}),
 (10295, 2022, '13431|13534', '2[6794]|86[95251]|58[95253]',
  {'rmedia': ('13431', '6794|95251|95253'), 'rmediana': ('13534', '6794|95251|95253')}),
 (10296, 2022, '13604', '2[6794]|86[95251]|386[9680,9681,9682,9692]',
  {'m12': ('13604', '6794|95251|9682'),
   'm14': ('13604', '6794|95251|9681'),
   'msem': ('13604', '6794|95251|9692'),
   'mtot': ('13604', '6794|95251|9680')}),
 (6805, 2022, '381', '11558[46292,46290]',
  {'desg': ('381', '46290'), 'dtot': ('381', '46292')}),
 (6803, 2022, '381', '1821[72129,72144]',
  {'dagua': ('381', '72144'), 'datot': ('381', '72129')}),
 (5938, 2022, '37', '', {'pib22': ('37', '')}),
 (5938, 2021, '37|498|513|517|6575|525', '',
  {'adm21': ('525', ''),
   'agro21': ('513', ''),
   'ind21': ('517', ''),
   'pib21': ('37', ''),
   'serv21': ('6575', ''),
   'vab21': ('498', '')}),
 (9509, 2022, '708|1606|10143', '',
  {'assal22': ('708', ''), 'salrs22': ('10143', ''), 'salsm22': ('1606', '')})]


def url_chamada(tabela, periodo, variaveis, classificacao):
    url = f"{API}/{tabela}/periodos/{periodo}/variaveis/{variaveis}?localidades=N6[N3[26]]"
    return url + (f"&classificacao={classificacao}" if classificacao else "")


def ler_resposta(resposta):
    """Resposta da API de agregados → {(variável, categorias): {código: texto}}."""
    saida = {}
    for var in resposta:
        for r in var["resultados"]:
            chave = "|".join(next(iter(c["categoria"])) for c in r["classificacoes"])
            saida[(str(var["id"]), chave)] = {
                s["localidade"]["id"]: next(iter(s["serie"].values())) for s in r["series"]}
    return saida


def bruto_da_api(obter_json):
    """Monta a tabela bruta a partir da API. `obter_json(url)` devolve o JSON já
    lido; o notebook passa uma função com `requests` e `timeout=120`."""
    colunas = {}
    for tabela, periodo, variaveis, classe, mapa in CHAMADAS:
        resp = ler_resposta(obter_json(url_chamada(tabela, periodo, variaveis, classe)))
        for col, (var, cat) in mapa.items():
            colunas[col] = resp[(var, cat)]
    locais = obter_json(API_LOCALIDADES)
    linhas = []
    for m in sorted(locais, key=lambda m: str(m["id"])):
        cod = str(m["id"])
        meso = MESO_API[m["microrregiao"]["mesorregiao"]["nome"]]
        linhas.append([cod, m["nome"], str(meso)] + [colunas[c][cod] for c in COLUNAS_BRUTAS[3:]])
    bruto = pd.DataFrame(linhas, columns=COLUNAS_BRUTAS)
    for c in COLUNAS_BRUTAS:
        if c != "nome":
            bruto[c] = pd.to_numeric(bruto[c], errors="coerce")
    return bruto


def indicadores(bruto):
    """Das tabelas do IBGE aos indicadores da aula, já arredondados como no CSV."""
    b = bruto
    d = pd.DataFrame({
        "cod_ibge": b["cod"].astype(int),
        "municipio": b["nome"],
        "mesorregiao": b["meso"].map(MESORREGIOES),
        "populacao": b["pop"].astype(int),
        "area_km2": b["area"].round(3),
        "densidade": (b["pop"] / b["area"]).round(2),
        "urbanizacao": b["urb"].round(2),
        "alfabetizacao": b["alfab"].round(2),
        "envelhecimento": b["env"].round(2),
        "idade_mediana": b["idmed"].astype(int),
        "renda_pc": b["rmedia"].round(2),
        "pobreza": (100 * (b["m14"] + b["msem"]) / b["mtot"]).round(2),
        "esgoto": (100 * b["desg"] / b["dtot"]).round(2),
        "agua": (100 * b["dagua"] / b["datot"]).round(2),
        "pib_pc": (1000 * b["pib22"] / b["pop"]).round(2),
        "agropecuaria": (100 * b["agro21"] / b["vab21"]).round(2),
        "industria": (100 * b["ind21"] / b["vab21"]).round(2),
        "servicos": (100 * b["serv21"] / b["vab21"]).round(2),
        "adm_publica": (100 * b["adm21"] / b["vab21"]).round(2),
        "emprego_formal": (100 * b["assal22"] / b["pop"]).round(2),
        "salario_medio": b["salsm22"].round(1),
    })
    return d[COLUNAS]


def obter_json(url, timeout=120):
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return r.json()


def carregar_dados():
    if os.path.exists(ARQ_CSV):
        return pd.read_csv(ARQ_CSV, encoding="utf-8"), "arquivo local"
    try:
        return indicadores(bruto_da_api(obter_json)), "API do IBGE"
    except Exception as erro:
        print("download falhou:", type(erro).__name__, str(erro)[:80])
    print("=" * 70)
    print("PLANO B: os municipios abaixo sao INVENTADOS. Mesmas colunas e os")
    print("mesmos cinco tipos de economia da base real; os valores sao")
    print("sinteticos. Os numeros NAO valem como resultado sobre Pernambuco.")
    print("=" * 70)
    return dados_sinteticos(ESP_PLANO_B), "PLANO B (sintetico)"


df, FONTE = carregar_dados()
print(f"fonte: {FONTE} | {len(df)} municipios, {df.shape[1]} colunas")
df.head()

In [ ]:
# Bloco 2 — um olhar antes de padronizar. Rode.
print(df[PERFIL].describe().T[["min", "50%", "max"]])
print()
print(df.loc[df["municipio"] == "Fernando de Noronha", ["municipio"] + PERFIL].T)

In [ ]:
# Bloco 2 — a matriz. COMPLETE.
#
# matriz() pega as colunas do perfil e aplica log nas tres de cauda longa;
# padronizar() deixa cada coluna com media 0 e desvio-padrao 1. Depois,
# compare a participacao de cada coluna na variancia total (que e quanto ela
# pesa na distancia) sem e com padronizacao.

def matriz(df, colunas=PERFIL, em_log=EM_LOG):
    """As colunas escolhidas, com log nas de cauda longa (ainda sem padronizar)."""
    X = df[colunas].astype(float).copy()
    for c in em_log:
        if c in X:
            X[c] = np.log(X[c])
    return X


# COMPLETE
Xm = ___                      # matriz com log
Z = ___                       # matriz padronizada

bruto = df[PERFIL].var(ddof=0)
participacao = pd.DataFrame({"como_esta_no_csv": 100 * bruto / bruto.sum(),
                             "padronizado": 100 * Z.var(ddof=0) / Z.var(ddof=0).sum()})
with pd.option_context("display.float_format", "{:.4f}".format):
    print(participacao.sort_values("como_esta_no_csv", ascending=False))

## Bloco 3 — componentes e o índice

O PCA procura a direção em que os municípios mais variam (1º componente), depois a de
maior variação no que sobra, perpendicular à primeira, e assim por diante. Cada
componente é uma soma ponderada dos indicadores padronizados; os pesos são as **cargas**.
O sinal de um componente é arbitrário: `pca()` o orienta para que a renda per capita
tenha carga positiva no 1º e a indústria no 2º.

In [ ]:
# Bloco 3 — o PCA. COMPLETE.

def pca(Z):
    """PCA sobre a matriz padronizada. As cargas do primeiro componente são
    orientadas para que a renda per capita tenha carga positiva (o sinal de um
    componente é arbitrário)."""
    p = PCA().fit(np.asarray(Z))
    cargas = pd.DataFrame(p.components_.T, index=Z.columns,
                          columns=[f"PC{i + 1}" for i in range(Z.shape[1])])
    escores = pd.DataFrame(p.transform(np.asarray(Z)), index=Z.index, columns=cargas.columns)
    ref = "renda_pc" if "renda_pc" in Z.columns else Z.columns[0]
    for c in cargas.columns[:2]:
        sinal = 1.0
        if c == "PC1" and cargas.loc[ref, c] < 0:
            sinal = -1.0
        if c == "PC2" and "industria" in Z.columns and cargas.loc["industria", c] < 0:
            sinal = -1.0
        cargas[c] *= sinal
        escores[c] *= sinal
    return {"variancia": p.explained_variance_ratio_, "cargas": cargas, "escores": escores}


# COMPLETE
res = ___                     # PCA da matriz padronizada Z

var = pd.Series(100 * res["variancia"], index=res["cargas"].columns)
print(pd.DataFrame({"% da variacao": var, "acumulada": var.cumsum()}).round(1))
print()
print(res["cargas"][["PC1", "PC2"]].sort_values("PC1").round(2))

### Responda

> Dê um nome a cada um dos dois primeiros componentes, olhando as cargas.

In [ ]:
# Bloco 3 — os municipios em dois eixos. Rode.
E = res["escores"]
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(E["PC1"], E["PC2"], s=14, color=AZUL, alpha=0.6)
for nome in ["Recife", "Fernando de Noronha", "Ipojuca", "Goiana", "Itapissuma",
             "Manari", "Petrolina", "Olinda", "Paranatama"]:
    i = df.index[df["municipio"] == nome]
    if len(i):
        ax.annotate(nome, (E.loc[i[0], "PC1"], E.loc[i[0], "PC2"]), fontsize=9)
ax.axhline(0, color="gray", lw=0.6); ax.axvline(0, color="gray", lw=0.6)
ax.set_xlabel("1o componente"); ax.set_ylabel("2o componente")
plt.show()

In [ ]:
# Bloco 3 — o indice de condicoes de vida. COMPLETE.
#
# 1o componente de cinco indicadores de quem MORA no municipio (INDICE):
# renda per capita (em log), pobreza, alfabetizacao, esgoto e agua.
# Confira o sinal: a pobreza tem de ter peso negativo, os outros positivo.

# COMPLETE
Zi = ___                      # matriz padronizada das colunas de INDICE (log so na renda)
ri = ___                      # PCA de Zi

print(f"o 1o componente guarda {100 * ri['variancia'][0]:.1f}% da variacao")
print("pesos:", ri["cargas"]["PC1"].round(2).to_dict())

indice = ri["escores"]["PC1"]
sinais = pd.Series([-1 if c == "pobreza" else 1 for c in INDICE], index=INDICE)
media_simples = (Zi * sinais).mean(axis=1)
tab = pd.DataFrame({
    "municipio": df["municipio"], "pib_pc": df["pib_pc"], "indice": indice.round(2),
    "pos_pib": df["pib_pc"].rank(ascending=False, method="first").astype(int),
    "pos_indice": indice.rank(ascending=False, method="first").astype(int),
    "pos_media": media_simples.rank(ascending=False, method="first").astype(int)})
print("\nos dez de maior PIB per capita:")
print(tab.sort_values("pos_pib").head(10).to_string(index=False))
print(f"\ncorrelacao entre o indice e a media simples: {np.corrcoef(indice, media_simples)[0, 1]:.3f}")
print(f"maior mudanca de posicao: {(tab['pos_indice'] - tab['pos_media']).abs().max()}")
print(f"correlacao de Spearman entre indice e PIB per capita: "
      f"{tab['pos_indice'].corr(tab['pos_pib']):.2f}")

### ✔ CHECKPOINT I

Ipojuca tem o maior PIB per capita de Pernambuco e fica longe do topo no índice de
condições de vida. Paranatama é 24º no PIB e está entre os últimos no índice.

Em duas frases: o que o PIB per capita de um município mede que o índice não mede? Qual
dos dois você usaria para distribuir um recurso social, e por quê?

## Bloco 4 — cinco grupos de municípios

Agora o K-means na matriz padronizada dos doze indicadores (`Z`, do Bloco 2).

In [ ]:
# Bloco 4 — quantos grupos? COMPLETE.
#
# Mesma curva do Bloco 1, agora com os municipios. Ha cotovelo? A silhueta
# tem pico?

# COMPLETE
curva_mun = ___

print(pd.DataFrame(curva_mun).T.round(3))
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
ks = list(curva_mun)
axs[0].plot(ks, [curva_mun[k]["inercia"] for k in ks], "o-", color=AZUL); axs[0].set_title("inercia")
axs[1].plot(ks, [curva_mun[k]["silhueta"] for k in ks], "o-", color=AZUL); axs[1].set_title("silhueta")
axs[1].set_ylim(0, 0.8)
plt.show()

In [ ]:
# Bloco 4 — os cinco grupos. COMPLETE.
#
# K-means com k = 5 na matriz Z (kmeans() ja usa 50 inicializacoes). Depois,
# a mediana de cada indicador em cada grupo e o numero de municipios.

# COMPLETE
km5 = ___
df["g"] = km5.labels_

perfil = df.groupby("g")[PERFIL + ["populacao"]].median()
perfil["n"] = df["g"].value_counts()
print(perfil.round(1).T)

In [ ]:
# Bloco 4 — os nomes. COMPLETE.
#
# Olhe o perfil acima e de um nome a cada grupo. Uma regra que funciona:
# o de maior industria, o de maior agropecuaria, o de maior renda entre os
# que sobram e, dos dois ultimos, o mais e o menos urbanizado. A funcao
# abaixo faz exatamente isso; compare com os nomes que voce daria.

def nomear_grupos(df, rotulos):
    """Dá nome aos cinco grupos pela mediana de cada um: o de maior indústria,
    o de maior agropecuária, o de maior renda entre os que sobram e, dos dois
    últimos, o mais e o menos urbanizado."""
    med = df.assign(g=np.asarray(rotulos)).groupby("g")[
        ["industria", "agropecuaria", "renda_pc", "urbanizacao"]].median()
    nomes = {}
    sobra = list(med.index)
    for nome, col in [("Enclaves industriais", "industria"),
                      ("Base agropecuária", "agropecuaria"),
                      ("Polos urbanos", "renda_pc")]:
        g = med.loc[sobra, col].idxmax()
        nomes[g] = nome
        sobra.remove(g)
    a, b = sobra
    urb, rur = (a, b) if med.loc[a, "urbanizacao"] > med.loc[b, "urbanizacao"] else (b, a)
    nomes[urb], nomes[rur] = "Interior urbanizado", "Interior rural"
    return pd.Series(np.asarray(rotulos), index=df.index).map(nomes)


# COMPLETE
df["grupo"] = ___

for g in GRUPOS:
    m = df[df["grupo"] == g].sort_values("populacao", ascending=False)["municipio"]
    print(f"{g:<22s} {len(m):3d}  {', '.join(m.head(6))}")
print()
print(pd.crosstab(df["grupo"], df["mesorregiao"]).loc[GRUPOS])

In [ ]:
# Bloco 4 — o mapa. Leia, nao precisa alterar.
#
# Contornos dos municipios (malha do IBGE, na versao GeoJSON do projeto
# geodata-br). Primeiro o arquivo Dados/mapa_municipios_pe.json; sem ele, a malha
# original no GitHub; sem rede, o notebook segue sem mapa.
ARQ_MAPA = "../Dados/mapa_municipios_pe.json"
URL_MAPA = ("https://raw.githubusercontent.com/tbrugz/geodata-br/master/"
            "geojson/geojs-26-mun.json")


def carregar_mapa():
    try:
        if os.path.exists(ARQ_MAPA):
            with open(ARQ_MAPA, encoding="utf-8") as f:
                g = json.load(f)
        else:
            g = requests.get(URL_MAPA, timeout=120).json()
    except Exception as erro:
        print("mapa indisponivel:", type(erro).__name__)
        return None
    contornos = {}
    for ft in g["features"]:
        p = ft["properties"]
        cod = int(p.get("cod_ibge", p.get("id", 0)))
        geo = ft["geometry"]
        partes = geo["coordinates"] if geo["type"] == "MultiPolygon" else [geo["coordinates"]]
        # na malha original, alguns municipios trazem partes separadas como
        # aneis do mesmo poligono: desenha-se cada anel
        contornos[cod] = [np.array(anel) for parte in partes for anel in parte]
    return contornos


def desenha_mapa(cor_por_codigo, titulo="", ax=None):
    """Pinta cada municipio com a cor dada; Fernando de Noronha fica de fora."""
    if CONTORNOS is None:
        print("(sem mapa nesta sessao)")
        return
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 4.6))
    polis, cores = [], []
    for cod, aneis in CONTORNOS.items():
        if cod in cor_por_codigo and cod != 2605459:
            for a in aneis:
                polis.append(a)
                cores.append(cor_por_codigo[cod])
    if not polis:
        print("(os codigos nao correspondem a municipios reais: sem mapa)")
        return None
    ax.add_collection(PolyCollection(polis, facecolors=cores, edgecolors="white",
                                     linewidths=0.3))
    ax.set_xlim(-41.45, -34.75); ax.set_ylim(-9.55, -7.2)
    ax.set_aspect(1 / np.cos(np.radians(8.3))); ax.axis("off")
    ax.set_title(titulo, loc="left")
    return ax


CONTORNOS = carregar_mapa()
print("mapa:", "pronto" if CONTORNOS else "indisponivel")

In [ ]:
# Bloco 4 — o mapa dos grupos. Rode.
COR_GRUPO = {"Polos urbanos": AZUL, "Enclaves industriais": LARANJA,
             "Base agropecuária": VERDE, "Interior urbanizado": CINZA,
             "Interior rural": CINZA_CLARO}
cores = dict(zip(df["cod_ibge"], df["grupo"].map(COR_GRUPO)))
ax = desenha_mapa(cores, "Cinco grupos de municipios (Fernando de Noronha fora do recorte)")
if ax is not None:
    for g in GRUPOS:
        ax.scatter([], [], s=80, marker="s", color=COR_GRUPO[g], label=g)
    ax.legend(loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.12), frameon=False)
    plt.show()

## Bloco 5 — os grupos resistem?

Três testes: a semente da inicialização, outro algoritmo e uma variável a menos.

In [ ]:
# Bloco 5 — sementes. COMPLETE.
#
# Rode o K-means com k = 5 cinquenta vezes, cada vez com UMA inicializacao so
# (n_init=1) e uma semente diferente. Guarde a inercia e o indice de Rand
# ajustado contra os cinco grupos do Bloco 4 (1 = mesma particao).

ref = km5.labels_
linhas = []
for s in range(50):
    # COMPLETE
    km = ___
    linhas.append({"semente": s, "inercia": km.inertia_,
                   "ari": adjusted_rand_score(km.labels_, ref)})
sem = pd.DataFrame(linhas)
print(sem[["inercia", "ari"]].describe().round(3))
print(f"\ninercia com 50 inicializacoes: {km5.inertia_:.1f}")

In [ ]:
# Bloco 5 — outro algoritmo: hierarquico (Ward). COMPLETE.
#
# linkage(..., method="ward") junta, a cada passo, os dois grupos cuja uniao
# menos aumenta a variacao interna. fcluster corta a arvore em 5 grupos.

def concordancia(a, b):
    """Fração de municípios no mesmo grupo, depois de casar os grupos."""
    t = pd.crosstab(np.asarray(a), np.asarray(b)).to_numpy()
    lin, col = linear_sum_assignment(-t)
    return float(t[lin, col].sum() / t.sum())


# COMPLETE
L = ___                         # linkage de Ward na matriz Z
ward = ___                      # corte em 5 grupos

fig, ax = plt.subplots(figsize=(12, 4.5))
dendrogram(L, truncate_mode="lastp", p=24, ax=ax, color_threshold=L[-5, 2] + 0.01)
ax.set_ylabel("distancia na juncao")
plt.show()
print(f"concordancia com o K-means: {100 * concordancia(ward, km5.labels_):.1f}%")
print(pd.crosstab(pd.Series(ward, name="Ward"), df["grupo"].rename("K-means")))

In [ ]:
# Bloco 5 — uma variavel a menos. COMPLETE.
#
# Tire um indicador de cada vez, refaca os cinco grupos e meca a concordancia
# com os grupos do Bloco 4.

resultado = {}
for c in PERFIL:
    colunas = [x for x in PERFIL if x != c]
    # COMPLETE
    km_c = ___
    resultado[NOMES[c]] = 100 * concordancia(km_c.labels_, km5.labels_)
print(pd.Series(resultado).sort_values().round(1))

### ✔ CHECKPOINT II

O governo do estado quer usar os cinco grupos para decidir onde investir primeiro em
saneamento.

Em duas frases: o que o agrupamento autoriza a afirmar sobre esses municípios, e o que não
autoriza? Você usaria os grupos ou o indicador de esgoto diretamente?